# Plan judge v2 on Colab (protocol 2, rank 1, one call per Plan)

1. Edit **only** the config cell below.
2. Run all cells. `MODE = "dry_run"` (default) validates inputs and shows the prompt without loading a model.
3. `MODE = "validate"` runs the **pilot** comparison of candidate judges on the reviewed case set (preliminary model selection, not definitive validation).
4. `MODE = "judge"` judges the selected variants. It resumes automatically; rerun the same config after a disconnect.

Requirements: a GPU runtime (L4 or A100 recommended; T4 needs `DTYPE = "float16"`), and a Colab Secret named `HF_TOKEN`
(🔑 sidebar) with access to any gated model (MedGemma requires accepting its license on Hugging Face first).
Inputs are never modified. Outputs go to `OUTPUT_DIR/<run-name>/<variant>/` on Drive; see `med_copilot/docs/plan_judging_v2.md`.

In [6]:
# ===================== CONFIG: edit only this cell =====================
MODE = "judge"                   # "dry_run" | "validate" | "judge"
REPO_URL = "https://github.com/ttogom/synthea-prep.git"   # public repository
REPO_REF = "plan-judge-v2"       # branch, tag, or commit to run (pin a commit for the full experiment)
DRIVE_ROOT = "/content/drive/MyDrive/synthea-prep"

# judge mode inputs (base or agentic run_experiments.py results.jsonl, read-only)
RESULTS_PATH = f"{DRIVE_ROOT}/runs/base/results.jsonl"
PATIENTS_PATH = "experiments/input/test_so.jsonl"       # relative paths are inside the cloned repo
LABELS_PATH = "experiments/input/test_labels.jsonl"
VARIANTS = ["llm_only"]          # one or more variant names, or ["all"]
GENERATOR_MODELS = None          # e.g. ["gpt_4o_mini"]; None judges every final model
OUTPUT_DIR = f"{DRIVE_ROOT}/plan_judge_v2_smoke"
RUN_NAME = None                  # None: <run directory>__<results sha12>
OWNER = "Alan"                   # your name, shown to teammates who hit your lease
WRITE_CSV = True
LIMIT = 20                       # e.g. 20 for a smoke test; None judges everything pending
TAKEOVER_STALE = False           # only after confirming nobody else is judging the variant

# judge model (chosen after the pilot)
BACKEND = "vllm"                 # "vllm" (guided JSON) or "transformers" (fallback)
JUDGE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"               # set after the pilot, e.g. "Qwen/Qwen3-4B-Instruct-2507"
JUDGE_REVISION = None            # None: resolve the current main to a commit and record it
DTYPE = "float16"                # use "float16" on T4 GPUs
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.9
CHAT_TEMPLATE_KWARGS = {}        # e.g. {"enable_thinking": False} for hybrid-reasoning Qwen3 checkpoints
ENGINE_ARGS = {}                 # e.g. {"limit_mm_per_prompt": {"image": 0}} for MedGemma (text only)
BATCH_SIZE = 64                  # Plans per checkpointed batch
MAX_ATTEMPTS = 3

# validate mode (pilot model comparison)
PILOT_CASES = "experiments/validation/plan_judge_v2_pilot_cases.jsonl"
PILOT_REPORTS = f"{DRIVE_ROOT}/plan_judge_v2_pilot"
PILOT_ALLOW_DRAFT = False        # True only for a dry pipeline check before labels are reviewed
CANDIDATES = [                   # run in order; a candidate that fails to load does not stop the others
    {"model": "Qwen/Qwen3-4B-Instruct-2507"},                  # ungated: run first
    {"model": "google/medgemma-4b-it", "engine_args": {"limit_mm_per_prompt": {"image": 0}}},  # gated
    # {"model": "microsoft/Phi-4-mini-instruct"},
]
# =======================================================================


## Setup (clone, install, GPU check)

In [7]:
import os, subprocess, sys
from pathlib import Path

if Path(DRIVE_ROOT).as_posix().startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

REPO = Path("/content/synthea-prep")
if not REPO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO)], check=True,
                   env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})   # LFS objects are not needed
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True, capture_output=True,
                        text=True).stdout.strip()
print("Repository commit:", COMMIT)
if MODE != "dry_run":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-colab-judge.txt")],
                   check=True)
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    print("GPU:", gpu.stdout.strip() or "none")
    if "T4" in gpu.stdout and DTYPE == "auto":
        print("WARNING: T4 GPUs lack bfloat16; set DTYPE = 'float16' in the config cell.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository commit: bbd502f7694fc19dcebd41737da752e69f48aaf2


## Credentials from Colab Secrets

In [8]:
# HF_TOKEN comes from Colab Secrets and is passed only to the child process environment.
CHILD_ENV = dict(os.environ)
if MODE != "dry_run":
    try:
        from google.colab import userdata
        CHILD_ENV["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets.")
    except Exception as exc:   # SecretNotFoundError, NotebookAccessError, or not on Colab
        print(f"No HF_TOKEN secret available ({type(exc).__name__}); only public, ungated models will load.")


## Run

In [9]:
sys.path.insert(0, str(REPO))
from scripts.plan_judge_v2.colab import judge_argv, validate_argv

def repo_path(path):
    return str(path) if Path(path).is_absolute() else str(REPO / path)

def run(argv):
    print("$ python judge_plans_v2.py", " ".join(argv))
    process = subprocess.Popen([sys.executable, "judge_plans_v2.py", *argv], cwd=REPO, env=CHILD_ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    print(f"exit status {code}")
    return code

if MODE in ("dry_run", "judge"):
    config = {"RESULTS_PATH": repo_path(RESULTS_PATH), "PATIENTS_PATH": repo_path(PATIENTS_PATH),
              "LABELS_PATH": repo_path(LABELS_PATH), "VARIANTS": VARIANTS, "GENERATOR_MODELS": GENERATOR_MODELS,
              "OUTPUT_DIR": OUTPUT_DIR, "RUN_NAME": RUN_NAME, "BACKEND": BACKEND, "JUDGE_MODEL": JUDGE_MODEL,
              "JUDGE_REVISION": JUDGE_REVISION, "DTYPE": DTYPE, "MAX_MODEL_LEN": MAX_MODEL_LEN,
              "GPU_MEMORY_UTILIZATION": GPU_MEMORY_UTILIZATION, "CHAT_TEMPLATE_KWARGS": CHAT_TEMPLATE_KWARGS,
              "ENGINE_ARGS": ENGINE_ARGS, "BATCH_SIZE": BATCH_SIZE, "MAX_ATTEMPTS": MAX_ATTEMPTS, "LIMIT": LIMIT,
              "WRITE_CSV": WRITE_CSV, "OWNER": OWNER, "TAKEOVER_STALE": TAKEOVER_STALE,
              "DRY_RUN": MODE == "dry_run"}
    EXIT = run(judge_argv(config))
elif MODE == "validate":
    # One process per candidate so GPU memory is released between models.
    codes = [run(validate_argv(candidate, cases=repo_path(PILOT_CASES), out_dir=PILOT_REPORTS, backend=BACKEND,
                               max_model_len=MAX_MODEL_LEN, allow_draft=PILOT_ALLOW_DRAFT))
             for candidate in CANDIDATES]
    EXIT = max(codes) if codes else 1
    if any(code == 0 for code in codes):
        run(["compare", "--reports", PILOT_REPORTS])
else:
    raise ValueError(f"Unknown MODE {MODE}")


$ python judge_plans_v2.py judge --results /content/drive/MyDrive/synthea-prep/runs/base/results.jsonl --patients /content/synthea-prep/experiments/input/test_so.jsonl --labels /content/synthea-prep/experiments/input/test_labels.jsonl --variants llm_only --out-dir /content/drive/MyDrive/synthea-prep/plan_judge_v2 --backend vllm --dtype float16 --max-model-len 4096 --gpu-memory-utilization 0.9 --chat-template-kwargs {} --batch-size 64 --max-attempts 3 --csv --dry-run
Protocol plan_judge_single_call v2; results sha256 805d2974b969; variants in file: llm_only, patient_retrieval, guidelines, patient_retrieval_guidelines
Denominator per variant: 2378 patients x 4 generators (gpt_4o_mini, ollama_gemma3_4b, ollama_llama3_2_3b, ollama_ministral_3_3b)
Note: 2581 patient(s) in test_so.jsonl have no results record; only patients in the results file are judged and counted.
[llm_only] 9512 rank-1 Plans by status: {"empty": 7, "pending": 9245, "unavailable": 260}; by reason: {"empty_generated_plan":

## Summary

In [10]:
import json
if MODE == "judge":
    roots = sorted(Path(OUTPUT_DIR).glob(f"{RUN_NAME or '*'}/*/manifest.json"))
    for path in roots:
        manifest = json.loads(path.read_text())
        print(f"{path.parent.name}: run {manifest['run_id'][:8]}, judge {manifest['judge']['model_id']}"
              f"@{(manifest['judge']['revision_resolved'] or '')[:10]}, counts {manifest.get('counts')}")
elif MODE == "validate" and (Path(PILOT_REPORTS) / "comparison.csv").exists():
    print((Path(PILOT_REPORTS) / "comparison.csv").read_text())
